<!-- WARNING: THIS FILE WAS AUTOGENERATED! DO NOT EDIT! -->

In [ ]:
from bioMONAI.data import *
from bioMONAI.transforms import *
from bioMONAI.transforms import ScaleIntensity
from bioMONAI.core import *
from bioMONAI.core import Path, set_determinism
from bioMONAI.data import get_image_files, get_target, RandomSplitter
from bioMONAI.nets import BasicUNet, DynUNet
from bioMONAI.losses import *
from bioMONAI.metrics import *

set_determinism(0)

In [ ]:
device = get_device()
print(device)

In [ ]:
bs, size = 16, 128

path = Path('../../bioMONAI_0/_data/Thunder_20230308/nuevos_datos/dataset')
path_x = path/'inputs'
path_y = path/'targets'

data_ops = {
    'blocks':       (BioImageBlock(cls=BioImage), BioImageBlock(cls=BioImage)),
    'get_items':    get_image_files,
    'get_y':        get_target(path_y, same_filename=True),
    'splitter':     RandomSplitter(valid_pct=0.2),
    'item_tfms':    [ScaleIntensity(), RandCrop2D(size), RandRot90(prob=0.5), RandFlip(prob=0.75)],
    'bs': bs,
}

data = get_dataloader(
    path_x, 
    show_summary=False,
    **data_ops,
    )

# print length of training and validation datasets
print('train images:', len(data.train_ds.items), '\nvalidation images:', len(data.valid_ds.items))

In [ ]:
data.show_batch(max_n=8, cmap='gray')

In [ ]:
from monai.networks.nets import BasicUNet, UNet # AttentionUnet, DynUNet, UNet, BasicUNet

In [ ]:
model = BasicUNet(spatial_dims=2, in_channels=1, out_channels=1)

In [ ]:
loss = MSSSIML1Loss(2, levels=2) #CombinedLoss(alpha=0, beta=0.5)
metrics = [SSIMMetric, MSELoss]

In [ ]:
trainer = fastTrainer(data, model, loss_fn=loss, metrics=metrics, show_summary=False)

In [ ]:
trainer.fit(2000)

In [ ]:
trainer.show_results(cmap='gray')

In [ ]:
import numpy as np
import torch
import matplotlib as plt

In [ ]:
image_path = '../../bioMONAI_0/_data/Thunder_20230308/nuevos_datos/dataset/inputs/sample_1.png'

image = Image.open(image_path).convert('L') 

array = np.array(image)

tensor = tensor1 = torch.from_numpy(array)

tensor = tensor.unsqueeze(0).unsqueeze(0)

tensor = tensor/255.0

tensor = tensor.to('cuda')

In [ ]:
tensor = tensor[:,:,0:256,0:256]

In [ ]:
output_tensor = model(tensor.to('cuda'))

In [ ]:
plt.imshow(array[0:256,0:256], cmap='gray')
plt.show()

In [ ]:
# Convertir el tensor de PyTorch a un arreglo de numpy

output_tensor = output_tensor.cpu()
output_array = output_tensor.detach().numpy()

In [ ]:
plt.imshow(output_array.squeeze((0,1)), cmap='gray')
plt.show()